<a href="https://colab.research.google.com/github/Takayoshi-code/My-important-data/blob/main/nanoGPT_Aozora_tiny.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## STEP1 nanoGPT環境をGitHubからダウンロードして自分の環境に展開します。

In [4]:
# ===== 安全版 =====

import os

print("===== Step 0: /content に移動 =====")
%cd /content

print("===== Step 1: クリーン =====")
!rm -rf nanoGPT
!rm -f master.zip

print("===== Step 2: ライブラリ =====")
!pip install -q sentencepiece datasets tqdm

print("===== Step 3: nanoGPT取得 =====")

!wget -q https://github.com/Takayoshi-code/nanoGPT-class/archive/refs/heads/master.zip
!unzip -q master.zip
!mv nanoGPT-class-master nanoGPT

print("===== Step 4: 移動 =====")
%cd /content/nanoGPT

print("===== Step 5: 確認 =====")
!ls

===== Step 0: /content に移動 =====
/content
===== Step 1: クリーン =====
===== Step 2: ライブラリ =====
===== Step 3: nanoGPT取得 =====
===== Step 4: 移動 =====
/content/nanoGPT
===== Step 5: 確認 =====
assets		 LICENSE		    scaling_laws.ipynb
bench.py	 model.py		    train.py
config		 nanoGPT_Aozora_tiny.ipynb  transformer_sizing.ipynb
configurator.py  README.md
data		 sample.py


## STEP2 作家、作品を選んでクリーニングして入力コーパス化

In [14]:
import io
import os
import re
import zipfile
import requests
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from urllib.parse import urljoin

CSV_ZIP_URL = "https://www.aozora.gr.jp/index_pages/list_person_all_extended_utf8.zip"
BASE_DIR = "/content/nanoGPT/aozora_selected"
RAW_OUTPUT = os.path.join(BASE_DIR, "selected_works.txt")
CLEAN_OUTPUT = os.path.join(BASE_DIR, "selected_works_clean.txt")
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; AozoraDownloader/1.0)"}

def normalize_name(s):
    if pd.isna(s):
        return ""
    return str(s).replace(" ", "").replace("　", "").strip()

def decode_aozora_text(raw):
    for enc in ["shift_jis", "cp932", "utf-8-sig", "utf-8"]:
        try:
            return raw.decode(enc)
        except UnicodeDecodeError:
            pass
    raise RuntimeError("文字コードを判定できませんでした。")

def clean_text(text):
    lines = text.split("\n")
    result = []
    started = False

    for line in lines:
        raw = line
        line = line.strip()

        if line.startswith("<|title|>"):
            continue
        if line.startswith("<|author|>"):
            continue

        if not started:
            if raw.startswith("　") and len(line) > 10 and "。" in line:
                started = True
            else:
                continue

        if (
            line.startswith("底本")
            or line.startswith("初出")
            or line.startswith("入力")
            or line.startswith("校正")
            or line.startswith("青空文庫")
            or line.startswith("作成ファイル")
            or "青空文庫作成ファイル" in line
        ):
            break

        if (
            "記号について" in line
            or "JIS" in line
            or "入力に使用" in line
            or "校正に使用" in line
        ):
            continue

        if re.fullmatch(r"[一二三四五六七八九十百千万]+", line):
            continue

        if re.fullmatch(r"[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+", line):
            continue

        if line in {"上", "中", "下", "前編", "後編", "序", "跋", "附録", "解説"}:
            continue

        line = re.sub(r"｜", "", line)
        line = re.sub(r"《.*?》", "", line)
        line = re.sub(r"［＃.*?］", "", line)
        line = re.sub(r"〔.*?〕", "", line)
        line = re.sub(r"（注.*?）", "", line)
        line = re.sub(r"（[\d０-９]+）", "", line)

        line = line.replace("／＼", "")
        line = line.replace("※", "")
        line = line.replace("＊", "")

        if re.fullmatch(r"[0-9０-９A-Za-z]+", line):
            continue

        if len(line) < 2:
            continue

        result.append(line)

    text = "\n".join(result)
    text = re.sub(r"\n{3,}", "\n\n", text)

    if text.count("。") < 10:
        return ""

    if len(text) < 1000:
        return ""

    return text.strip()

def download_work(card_url):
    r = requests.get(card_url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    r.encoding = r.apparent_encoding
    soup = BeautifulSoup(r.text, "html.parser")

    zip_candidates = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().split("?")[0].endswith(".zip"):
            url = urljoin(card_url, href)
            label = a.get_text(" ", strip=True)
            priority = 0 if "テキスト" in label else 1
            zip_candidates.append((priority, url))

    if not zip_candidates:
        return None

    zip_candidates.sort(key=lambda x: x[0])

    for _, zip_url in zip_candidates:
        try:
            r = requests.get(zip_url, headers=HEADERS, timeout=60)
            r.raise_for_status()

            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                txt_files = [name for name in z.namelist() if name.lower().endswith(".txt")]

                if not txt_files:
                    continue

                raw = z.read(txt_files[0])
                return decode_aozora_text(raw)

        except Exception:
            continue

    return None

def main():
    os.makedirs(BASE_DIR, exist_ok=True)

    print("青空文庫の作品一覧を取得しています...")

    r = requests.get(CSV_ZIP_URL, headers=HEADERS, timeout=60)
    r.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        csv_files = [name for name in z.namelist() if name.lower().endswith(".csv")]
        if not csv_files:
            raise RuntimeError("CSVが見つかりません。")
        with z.open(csv_files[0]) as f:
            df = pd.read_csv(f, low_memory=False)

    print(f"登録データ数: {len(df):,}")

    df["作家名"] = (
        df["姓"].fillna("").astype(str).str.strip()
        + " "
        + df["名"].fillna("").astype(str).str.strip()
    ).str.strip()

    df["作家名検索用"] = df["作家名"].apply(normalize_name)

    keyword = input("作家名を入力してください: ").strip()
    keyword_normalized = normalize_name(keyword)

    mask = df["作家名検索用"].str.contains(
        keyword_normalized,
        na=False,
        regex=False
    )

    authors = (
        df.loc[mask, ["人物ID", "作家名"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    if len(authors) == 0:
        raise ValueError(f"「{keyword}」に該当する作家が見つかりません。")

    print("\n=== 作家候補 ===")

    for i, row in authors.iterrows():
        print(f"{i:3d}: {row['作家名']} (人物ID={row['人物ID']})")

    if len(authors) == 1:
        n = 0
        print("候補が1人なので自動選択します。")
    else:
        while True:
            try:
                n = int(input("左端の選択番号を入力してください: "))
                if 0 <= n < len(authors):
                    break
            except ValueError:
                pass
            print(f"0～{len(authors)-1}を入力してください。")

    author_id = authors.iloc[n]["人物ID"]
    author_name = authors.iloc[n]["作家名"]

    print(f"\n選択した作家: {author_name} (人物ID={author_id})")

    columns = ["作品ID", "作品名", "図書カードURL"]

    if "副題" in df.columns:
        columns.insert(2, "副題")

    works = (
        df.loc[df["人物ID"] == author_id, columns]
        .drop_duplicates(subset=["作品ID"])
        .reset_index(drop=True)
    )

    print(f"\n=== {author_name} の作品一覧（{len(works)}作品）===")

    for i, row in works.iterrows():
        title = str(row["作品名"])

        if "副題" in works.columns and pd.notna(row["副題"]):
            subtitle = str(row["副題"]).strip()
            if subtitle:
                title += " ― " + subtitle

        print(f"{i:3d}: {title} (作品ID={row['作品ID']})")

    print("\n複数選択できます。")
    print("例: 0,3,5,8")
    print("全作品: all")

    selection = input("作品番号を入力してください: ").strip()

    if selection.lower() == "all":
        selected_indices = list(range(len(works)))
    else:
        try:
            selected_indices = [int(x.strip()) for x in selection.split(",")]
        except ValueError:
            raise ValueError("作品番号は 0,3,5 の形式で入力してください。")

    selected_indices = list(dict.fromkeys(selected_indices))

    for i in selected_indices:
        if i < 0 or i >= len(works):
            raise ValueError(f"作品番号 {i} は範囲外です。")

    print(f"\n{len(selected_indices)}作品をダウンロードします。")

    downloaded = []

    for number, i in enumerate(selected_indices, 1):
        work = works.iloc[i]
        title = str(work["作品名"])
        card_url = str(work["図書カードURL"])

        print(f"[{number}/{len(selected_indices)}] {title}")

        try:
            text = download_work(card_url)
        except Exception as e:
            print(f"  → ERROR: {e}")
            continue

        if text is None:
            print("  → TXTを取得できませんでした。")
            continue

        downloaded.append({
            "title": title,
            "text": text
        })

        print(f"  → OK {len(text):,} chars")

    if not downloaded:
        raise RuntimeError("作品を1件も取得できませんでした。")

    print(f"\nRAWファイルを作成: {RAW_OUTPUT}")

    with open(RAW_OUTPUT, "w", encoding="utf-8") as out:
        for item in downloaded:
            out.write(f"<|title|>{item['title']}\n")
            out.write(f"<|author|>{author_name}\n")
            out.write(item["text"])
            out.write("\n\n<|endoftext|>\n\n")

    original_size = os.path.getsize(RAW_OUTPUT)

    print("\nクリーニングを開始します...")

    total = len(downloaded)
    kept = 0
    total_chars = 0

    with open(CLEAN_OUTPUT, "w", encoding="utf-8") as out:
        for item in tqdm(downloaded):
            cleaned = clean_text(item["text"])

            if cleaned:
                chars = len(cleaned)
                kept += 1
                total_chars += chars

                print(f"Work {kept:2d}: {item['title']} : {chars:,} chars")

                out.write(cleaned)
                out.write("\n\n<|endoftext|>\n\n")
            else:
                print(f"REMOVED: {item['title']}")

    cleaned_size = os.path.getsize(CLEAN_OUTPUT)

    print()
    print("====================================")
    print(f"Author           : {author_name}")
    print(f"Selected works   : {len(selected_indices)}")
    print(f"Downloaded works : {total}")
    print(f"Kept works       : {kept}")
    print(f"Removed works    : {total-kept}")
    print("------------------------------------")
    print(f"Total chars      : {total_chars:,}")
    print(f"Original size    : {original_size/1024:.1f} KB")
    print(f"Cleaned size     : {cleaned_size/1024:.1f} KB")

    if original_size > 0:
        print(f"Reduction        : {100*(1-cleaned_size/original_size):.1f}%")

    print("------------------------------------")
    print(f"RAW output       : {RAW_OUTPUT}")
    print(f"Clean output     : {CLEAN_OUTPUT}")
    print("====================================")

if __name__ == "__main__":
    main()

青空文庫の作品一覧を取得しています...
登録データ数: 19,502
作家名を入力してください: 田山花袋

=== 作家候補 ===
  0: 田山 花袋 (人物ID=214)
候補が1人なので自動選択します。

選択した作家: 田山 花袋 (人物ID=214)

=== 田山 花袋 の作品一覧（153作品）===
  0: 赤い鳥居 (作品ID=48809)
  1: アカシヤの花 (作品ID=48969)
  2: 秋の岐蘇路 (作品ID=59179)
  3: 朝 (作品ID=4598)
  4: あさぢ沼 (作品ID=48811)
  5: 新しい生 (作品ID=48812)
  6: あちこちの渓谷 (作品ID=48995)
  7: 雨の日に (作品ID=48814)
  8: 或新年の小説評 (作品ID=48815)
  9: ある僧の奇蹟 (作品ID=43269)
 10: ある時に (作品ID=48816)
 11: ある日 (作品ID=48819)
 12: ある日の印旛沼 (作品ID=48820)
 13: アンナ、パブロオナ (作品ID=48970)
 14: 磯清水 (作品ID=48822)
 15: 一少女 (作品ID=48823)
 16: 一夜のうれい (作品ID=61299)
 17: 一国の首都 (作品ID=48693)
 18: 一室 (作品ID=48824)
 19: 行つて見たいところ (作品ID=48996)
 20: 一兵卒 (作品ID=1066)
 21: 田舎からの手紙 (作品ID=48825)
 22: 田舎教師 (作品ID=1668)
 23: 『田舎教師』について (作品ID=46594)
 24: 犬 (作品ID=48827)
 25: 伊良湖岬 (作品ID=48828)
 26: 海をわたる (作品ID=48830)
 27: エンジンの響 (作品ID=48990)
 28: 大阪で (作品ID=48831)
 29: 丘の上の家 ― 抄 (作品ID=58938)
 30: 隠岐がよひの船 (作品ID=48999)
 31: 尾崎紅葉とその作品 (作品ID=48832)
 32: 女と情と愛と (作品ID=48833)
 33: 女の温泉 (作品ID=49003)
 34: 間居 (作品ID=48835

100%|██████████| 2/2 [00:00<00:00, 288.62it/s]

Work  1: 蒲団 : 45,124 chars
Work  2: 『蒲団』を書いた頃 : 2,513 chars

Author           : 田山 花袋
Selected works   : 2
Downloaded works : 2
Kept works       : 2
Removed works    : 0
------------------------------------
Total chars      : 47,637
Original size    : 161.6 KB
Cleaned size     : 138.5 KB
Reduction        : 14.3%
------------------------------------
RAW output       : /content/nanoGPT/aozora_selected/selected_works.txt
Clean output     : /content/nanoGPT/aozora_selected/selected_works_clean.txt


## STEP3 SentencePiece（5000語）規模のトークナイザを動かしinput.textに登場するトークンを学習する。

In [15]:
# ============================================================
# 日本語コーパス → SentencePiece → train.bin（最終完成版）
# ============================================================

import os
import re
import sentencepiece as spm
import numpy as np
import pickle
import threading
import time

BASE = "/content/nanoGPT"
INPUT = "/content/nanoGPT/aozora_selected/selected_works_clean.txt"
MODEL_PREFIX = "/content/nanoGPT/jp"
OUT_DIR = os.path.join(BASE, "jp")

# =========================
# 全体タイマー
# =========================
total_start = time.time()

# =========================
# 進捗表示
# =========================
running = True
def heartbeat():
    flag = True
    while running:
        print("ただいま計算中..." if flag else "　　　　　　　　", flush=True)
        flag = not flag
        time.sleep(5)

t = threading.Thread(target=heartbeat)
t.start()

try:
    # =========================
    # ① モデル削除
    # =========================
    t0 = time.time()
    for ext in [".model", ".vocab"]:
        path = MODEL_PREFIX + ext
        if os.path.exists(path):
            os.remove(path)
    print(f"[STEP1] モデル削除時間: {time.time() - t0:.2f} 秒")

    # =========================
    # ② tokenizer学習
    # =========================
    print("=== SentencePiece Training ===")
    t0 = time.time()

    spm.SentencePieceTrainer.train(
        input=INPUT,
        model_prefix=MODEL_PREFIX,
        vocab_size=5000,
        character_coverage=0.9995,
        model_type='unigram',
        num_threads=8,
        input_sentence_size=2000000,   # ←増やした
        shuffle_input_sentence=True,
        max_sentence_length=4096,
        hard_vocab_limit=False,
        user_defined_symbols=["<|endoftext|>"]
    )

    print(f"[STEP2] Tokenizer学習時間: {time.time() - t0:.2f} 秒")
    print("Tokenizer DONE")

    # =========================
    # ③ トークナイズ（最重要部分）
    # =========================
    print("=== Encoding ===")
    t0 = time.time()

    sp = spm.SentencePieceProcessor()
    sp.load(MODEL_PREFIX + ".model")

    ids = []
    eot_id = sp.piece_to_id("<|endoftext|>")

    # 全読み込み
    with open(INPUT, encoding="utf-8") as f:
        data = f.read()

    # 元chunk分割
    chunks = data.split("<|endoftext|>")

    # -------------------------
    # 文単位に細分化（超重要）
    # -------------------------
    new_chunks = []

    for chunk in chunks:
        chunk = chunk.strip()
        if not chunk:
            continue

        # 文分割（精度UP）
        sentences = re.split(r"[。！？]", chunk)

        for s in sentences:
            s = s.strip()
            if len(s) > 50:
                new_chunks.append(s + "。")

    # -------------------------
    # encode
    # -------------------------
    for chunk in new_chunks:
        ids.extend(sp.encode(chunk))
        ids.append(eot_id)

    print("Total tokens:", len(ids))
    print(f"[STEP3] Encoding時間: {time.time() - t0:.2f} 秒")

    # =========================
    # ④ train / val 分割
    # =========================
    t0 = time.time()

    n = int(len(ids) * 0.9)
    train_ids = np.array(ids[:n], dtype=np.uint16)
    val_ids   = np.array(ids[n:], dtype=np.uint16)

    print(f"[STEP4] 分割時間: {time.time() - t0:.2f} 秒")

    # =========================
    # ⑤ 保存
    # =========================
    t0 = time.time()

    os.makedirs(OUT_DIR, exist_ok=True)

    train_ids.tofile(os.path.join(OUT_DIR, "train.bin"))
    val_ids.tofile(os.path.join(OUT_DIR, "val.bin"))

    with open(os.path.join(OUT_DIR, "meta.pkl"), "wb") as f:
        pickle.dump({"vocab_size": sp.get_piece_size()}, f)

    print(f"[STEP5] 保存時間: {time.time() - t0:.2f} 秒")

    print("=== ALL DONE ===")
    print("vocab_size =", sp.get_piece_size())

finally:
    running = False
    t.join()

# =========================
# 総時間
# =========================
print(f"\n=== 総処理時間: {time.time() - total_start:.2f} 秒 ===")

ただいま計算中...
[STEP1] モデル削除時間: 0.00 秒
=== SentencePiece Training ===
[STEP2] Tokenizer学習時間: 0.15 秒
Tokenizer DONE
=== Encoding ===
Total tokens: 11756
[STEP3] Encoding時間: 0.01 秒
[STEP4] 分割時間: 0.00 秒
[STEP5] 保存時間: 0.00 秒
=== ALL DONE ===
vocab_size = 5000

=== 総処理時間: 5.00 秒 ===


## STEP4 入力コーパスを学習用と検証用の２つに分けます。

In [16]:
import sentencepiece as spm
import numpy as np
import os
import pickle

print("=== SentencePiece Training ===")


BASE = "/content/nanoGPT"
sp = spm.SentencePieceProcessor()
sp.load(os.path.join(BASE, "jp.model"))


input_file = "/content/nanoGPT/aozora_selected/selected_works.txt"
#out_dir = os.path.join(BASE, "jp")
out_dir = os.path.join(BASE, "data", "jp")

os.makedirs(out_dir, exist_ok=True)

print("Reading...")
ids = []
with open(input_file, encoding="utf-8") as f:
    for line in f:
        ids.extend(sp.encode(line))   # ← メモリ安全

print("Encoding done")
print("Total tokens:", len(ids))

n = int(0.9 * len(ids))
train_ids = np.array(ids[:n], dtype=np.uint16)
val_ids = np.array(ids[n:], dtype=np.uint16)

train_ids.tofile(os.path.join(out_dir, "train.bin"))
val_ids.tofile(os.path.join(out_dir, "val.bin"))

with open(os.path.join(out_dir, "meta.pkl"), "wb") as f:
    pickle.dump({"vocab_size": sp.get_piece_size()}, f)
print(f"train tokens = {len(train_ids):,}")
print(f"val tokens   = {len(val_ids):,}")

print("DONE")

=== SentencePiece Training ===
Reading...
Encoding done
Total tokens: 30893
train tokens = 27,803
val tokens   = 3,090
DONE


## STEP5 GPUが使えるかをチェックします。

In [17]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## STEP 6 Transformerで学習させます。　CPU環境では --device=cpu とし、 GPUが使えるなら　--device=cuda にする。

In [ ]:
## 人間失格規模用

!python train.py \
  --dataset=jp \
  --device=cuda \
  --compile=False \
  --init_from=scratch \
  --n_layer=22 \
  --n_head=16 \
  --n_embd=128 \
  --batch_size=8 \
  --gradient_accumulation_steps=1 \
  --block_size=1024 \
  --max_iters=8000 \
  --lr_decay_iters=8000 \
  --warmup_iters=20 \
  --learning_rate=4e-4 \
  --min_lr=3e-5 \
  --eval_interval=20 \
  --eval_iters=20 \
  --dropout=0.1 \
  --log_interval=10 \
  --dtype=float16


Overriding: dataset = jp
Overriding: device = cuda
Overriding: compile = False
Overriding: init_from = scratch
Overriding: n_layer = 22
Overriding: n_head = 16
Overriding: n_embd = 128
Overriding: batch_size = 8
Overriding: gradient_accumulation_steps = 1
Overriding: block_size = 1024
Overriding: max_iters = 8000
Overriding: lr_decay_iters = 8000
Overriding: warmup_iters = 20
Overriding: learning_rate = 0.0004
Overriding: min_lr = 3e-05
Overriding: eval_interval = 20
Overriding: eval_iters = 20
Overriding: dropout = 0.1
Overriding: log_interval = 10
Overriding: dtype = float16
tokens per iteration will be: 8,192
found vocab_size = 5000 (inside data/jp/meta.pkl)
Initializing a new model from scratch
number of parameters: 4.97M
/content/nanoGPT/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
num decayed parameter tensors: 90, with

In [ ]:
## 走れメロス規模用
!python train.py \
  --dataset=jp \
  --device=cuda \
  --compile=False \
  --init_from=scratch \
  --n_layer=2 \
  --n_head=2 \
  --n_embd=128 \
  --batch_size=4 \
  --gradient_accumulation_steps=1 \
  --block_size=128 \
  --max_iters=6000 \
  --lr_decay_iters=6000 \
  --warmup_iters=20 \
  --learning_rate=3e-4 \
  --min_lr=3e-5 \
  --eval_interval=20 \
  --eval_iters=20 \
  --dropout=0.1 \
  --log_interval=10 \
  --dtype=float16


Overriding: dataset = jp
Overriding: device = cuda
Overriding: compile = False
Overriding: init_from = scratch
Overriding: n_layer = 2
Overriding: n_head = 2
Overriding: n_embd = 128
Overriding: batch_size = 4
Overriding: gradient_accumulation_steps = 1
Overriding: block_size = 128
Overriding: max_iters = 6000
Overriding: lr_decay_iters = 6000
Overriding: warmup_iters = 20
Overriding: learning_rate = 0.0003
Overriding: min_lr = 3e-05
Overriding: eval_interval = 20
Overriding: eval_iters = 20
Overriding: dropout = 0.1
Overriding: log_interval = 10
Overriding: dtype = float16
tokens per iteration will be: 512
found vocab_size = 5000 (inside data/jp/meta.pkl)
Initializing a new model from scratch
number of parameters: 1.03M
/content/nanoGPT/train.py:196: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))
num decayed parameter tensors: 10, with 1,04

## STEP7 文章生成をします。

In [ ]:
# generate_sp.py
import os
import torch
import sentencepiece as spm
from contextlib import nullcontext
os.chdir("/content/nanoGPT")
from model import GPTConfig, GPT
# =========================
# 設定
# =========================
out_dir = 'out'
#sp_model_path = 'jp16k.model'
#sp_model_path = "/home/yokota/DAZAI_LLM/nanoGPT/jp.model"
sp_model_path = "/content/nanoGPT/jp.model"


prompt = "女は、甲州の"   # ← ここで文書の方向性を決める
num_samples = 1
max_new_tokens = 200   # ← 長文
#temperature = 0.9     # ← 安定寄り
#top_k = 20
device = 'cuda' if torch.cuda.is_available() else 'cpu'

repetition_penalty = 1.1
temperature = 0.2
top_k = 20
top_p = 0.9
# =========================
# SentencePiece
# =========================
sp = spm.SentencePieceProcessor()
sp.load(sp_model_path)

def encode(s): return sp.encode(s, out_type=int)
def decode(l): return sp.decode(l)

# =========================
# モデル
# =========================
ckpt_path = os.path.join(out_dir, 'ckpt.pt')
checkpoint = torch.load(ckpt_path, map_location=device)

model = GPT(GPTConfig(**checkpoint['model_args']))
state_dict = checkpoint['model']

# compile対策
for k in list(state_dict.keys()):
    if k.startswith('_orig_mod.'):
        state_dict[k[len('_orig_mod.'):]] = state_dict.pop(k)

model.load_state_dict(state_dict)
model.to(device)
model.eval()

ctx = nullcontext()

# =========================
# 生成（改良版）
# =========================
# =========================
# 生成（最終版・完成）
# =========================
def generate(model, idx, max_new_tokens):

    block_size = model.config.block_size
    last_tokens = []  # ← 追加（直近トークン履歴）

    for _ in range(max_new_tokens):

        idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]
        logits, _ = model(idx_cond)

        logits = logits[:, -1, :] / temperature

        # -------------------------
        # repetition penalty（logitsに直接適用）
        # -------------------------
        for token in set(idx[0].tolist()):
            logits[0][token] /= repetition_penalty

        # -------------------------
        # top_k
        # -------------------------
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = -float('Inf')

        # -------------------------
        # top_p（追加）
        # -------------------------
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)

            sorted_indices_to_remove = cumulative_probs > top_p
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[0][indices_to_remove] = -float('Inf')

        probs = torch.softmax(logits, dim=-1)

        # -------------------------
        # last_tokens repetition防止（最重要）
        # -------------------------
        for _ in range(10):  # 最大10回リトライ
            next_token = torch.multinomial(probs, num_samples=1)

            if next_token.item() not in last_tokens:
                break

        # 履歴更新
        last_tokens.append(next_token.item())
        if len(last_tokens) > 10:
            last_tokens.pop(0)

        idx = torch.cat((idx, next_token), dim=1)

    return idx
# =========================
# 実行
# =========================
start_ids = encode(prompt)
x = torch.tensor(start_ids, dtype=torch.long, device=device)[None, ...]

with torch.no_grad():
    with ctx:
        for _ in range(num_samples):
            y = generate(model, x, max_new_tokens)
            text = decode(y[0].tolist())

            print("==========")
            print(text)

number of parameters: 4.97M
女は、甲州の生れで二十八歳でした。五つになる女児と、高円寺のアパートに住んでいました。夫と死別して、三年になると言っていました。 「あなたは、ずいぶん苦労して育って来たみたいなひとね。よく気がきくわ。可哀そうに」 はじめて、男めかけみたいな生活をしました。シヅ子(というのが、その女記者の名前でした)が新宿の雑誌社に勤めに出たあとは、自分とそれからシゲ子という五つの女児と二人、おとなしくお留守番という事になりました。それまでは、母の留守には、シゲ子はアパートの管理人の部屋で遊んでいたようでしたが、「気のきく」おじさんが遊び相手として現われたので、大いに御機嫌がいい様子でした。 一週間ほど、ぼんやり、自分はそこにいました。アパートの窓のすぐ近くの電線に、奴凧が一つひっからまっていて、春のほこり風に吹かれ、破られ、それでもなかなか、しつっこく電線にからみついて離れず、何やら首肯いたりなんかしているので、自分はそれを見る度毎に苦笑し、
